In [1]:
# Optional installs
# Run this only if the packages are missing in your environment.
# !pip install pymupdf pdfplumber python-dotenv tqdm

In [2]:
# ── Imports and logging ─────────────────────────────────────
import os
import re
import json
import logging
import urllib.request
import urllib.error
import time
import random
from pathlib import Path
from typing import Optional

import pdfplumber
from dotenv import load_dotenv, find_dotenv
from tqdm import tqdm

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

# ── ENV LOADING: same style as generation notebook ──────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

print("Imports ready")
print("PyMuPDF available:", fitz is not None)

Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Imports ready
PyMuPDF available: True


In [3]:
# ── Configuration ───────────────────────────────────────────
# Update these paths for your local machine if needed.

PDF_PATH_MANUAL = None

RAW_OUTPUT_PATH = Path("emirates_nbd_style_reference_raw.json")
CLEAN_OUTPUT_PATH = Path("emirates_nbd_style_reference_clean.json")
OUTPUT_PATH = CLEAN_OUTPUT_PATH
DEBUG = True

def _normalise_pdf_path(value):
    if value is None:
        return None
    p = Path(str(value)).expanduser()
    return p if p.exists() and p.suffix.lower() == ".pdf" else None

def find_emirates_pdf() -> Path | None:
    """
    Find the Emirates NBD IFRS S1/S2 PDF from:
    - manual override
    - .env variable EMIRATES_NBD_PDF_PATH
    - common nearby paths
    - recursive search in current project folders
    """
    # 1) Manual override
    manual = _normalise_pdf_path(PDF_PATH_MANUAL)
    if manual:
        print("Using manual PDF path:", manual)
        return manual

    # 2) Environment variable
    env_path = _normalise_pdf_path(os.getenv("EMIRATES_NBD_PDF_PATH"))
    if env_path:
        print("Using EMIRATES_NBD_PDF_PATH:", env_path)
        return env_path

    # 3) Exact common paths
    exact_names = [
        "emirates_nbd_group_2024_ifrs_s1_s2.pdf",
        "emirates_nbd_2024_ifrs_s1_s2.pdf",
        "Emirates NBD Group 2024 IFRS S1 S2.pdf",
        "Emirates_NBD_Group_2024_IFRS_S1_S2.pdf",
        "emirates_nbd_group_2024_sustainability_report.pdf",
    ]

    roots = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "reports",
        Path.cwd() / "reference_reports",
        Path.cwd() / "gen_data",
        Path.cwd() / "gen_data" / "reports",
        Path.cwd() / "gen_data" / "reference_reports",
        Path.cwd().parent,
        Path.cwd().parent / "data",
        Path.cwd().parent / "reports",
        Path.cwd().parent / "reference_reports",
        Path.cwd().parent / "gen_data",
        Path.cwd().parent / "gen_data" / "reports",
        Path.cwd().parent / "gen_data" / "reference_reports",
        Path("/mnt/data"),
    ]

    checked_exact = []
    for root in roots:
        for name in exact_names:
            candidate = root / name
            checked_exact.append(candidate)
            if candidate.exists() and candidate.suffix.lower() == ".pdf":
                print("Found PDF:", candidate)
                return candidate

    # 4) Recursive search by patterns
    patterns = [
        "*emirates*nbd*.pdf",
        "*Emirates*NBD*.pdf",
        "*ifrs*s1*s2*.pdf",
        "*IFRS*S1*S2*.pdf",
    ]

    recursive_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data"),
    ]

    matches = []
    for root in recursive_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            try:
                matches.extend(root.rglob(pattern))
            except Exception as exc:
                print(f"Could not recursively search {root}: {exc}")

    # De-duplicate and prefer Emirates NBD names
    unique_matches = []
    seen = set()
    for p in matches:
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen and p.is_file() and p.suffix.lower() == ".pdf":
            seen.add(key)
            unique_matches.append(p)

    if unique_matches:
        unique_matches = sorted(
            unique_matches,
            key=lambda p: (
                0 if "emirates" in p.name.lower() and "nbd" in p.name.lower() else 1,
                len(str(p))
            )
        )
        print("Found PDF by recursive search:", unique_matches[0])
        print("Other PDF matches:", [str(p) for p in unique_matches[1:5]])
        return unique_matches[0]

    print("No Emirates NBD PDF found automatically.")
    print("Checked common exact paths such as:")
    for p in checked_exact[:12]:
        print(" -", p)
    print("Set PDF_PATH_MANUAL in this cell or add EMIRATES_NBD_PDF_PATH to your .env.")
    return None

PDF_PATH = find_emirates_pdf()

# ── Azure OpenAI REST URL settings: same logic as generation notebook ──
# The generation notebook uses full role-based Azure deployment URLs:
# AZURE_OPENAI_WRITER_URL, AZURE_OPENAI_JUDGE_URL, AZURE_OPENAI_REVISER_URL.
#
# This extraction notebook uses one URL for style extraction.
# Since you want GPT-5.2, the cleanest option is to reuse your Judge URL:
# AZURE_OPENAI_STYLE_URL=<full GPT-5.2 chat completions URL>
# or simply rely on AZURE_OPENAI_JUDGE_URL if it already points to GPT-5.2.
#
# Required if all deployments are in the same Azure resource:
# AZURE_OPENAI_API_KEY=<shared Azure resource key>
# AZURE_OPENAI_STYLE_URL=<full GPT-5.2 deployment URL>
#
# Manual override examples:
# AZURE_OPENAI_API_KEY_MANUAL = "paste-key-here"
# AZURE_OPENAI_STYLE_URL_MANUAL = "https://<resource>.openai.azure.com/openai/deployments/<gpt-5.2-deployment>/chat/completions?api-version=2024-10-21"

AZURE_OPENAI_API_KEY_MANUAL = None
AZURE_OPENAI_STYLE_URL_MANUAL = None

def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")

AZURE_OPENAI_API_KEY = (
    AZURE_OPENAI_API_KEY_MANUAL
    or os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("AZURE_OPENAI_STYLE_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
)

# Style URL first, then Judge URL because Judge is GPT-5.2 in your generation notebook.
AZURE_OPENAI_STYLE_URL = _clean_url(
    AZURE_OPENAI_STYLE_URL_MANUAL
    or os.getenv("AZURE_OPENAI_STYLE_URL")
    or os.getenv("AZURE_OPENAI_GPT52_URL")
    or os.getenv("AZURE_OPENAI_GPT_5_2_URL")
    or os.getenv("AZURE_OPENAI_JUDGE_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)

def validate_style_config() -> None:
    required = {
        "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
        "AZURE_OPENAI_STYLE_URL or AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_STYLE_URL,
    }

    missing = [name for name, value in required.items() if not value]
    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "style_url_loaded": bool(AZURE_OPENAI_STYLE_URL),
            "judge_url_loaded": bool(os.getenv("AZURE_OPENAI_JUDGE_URL")),
            "chat_url_loaded": bool(os.getenv("AZURE_OPENAI_CHAT_URL")),
        }
        raise ValueError(
            "Missing Azure REST configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags (keys are never printed):\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nUse the same style as the generation notebook:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_STYLE_URL=<full GPT-5.2 deployment URL>\n\n"
              "Or, if your generation notebook already has GPT-5.2 judge configured:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_JUDGE_URL=<full GPT-5.2 deployment URL>\n"
        )

    if not AZURE_OPENAI_STYLE_URL.startswith("https://"):
        raise ValueError(
            "AZURE_OPENAI_STYLE_URL must be a full HTTPS Azure deployment URL, "
            f"got: {AZURE_OPENAI_STYLE_URL!r}"
        )

validate_style_config()

print("PDF_PATH:", PDF_PATH)
print("RAW_OUTPUT_PATH:", RAW_OUTPUT_PATH.resolve())
print("CLEAN_OUTPUT_PATH:", CLEAN_OUTPUT_PATH.resolve())
print("DEBUG:", DEBUG)
print("Azure REST style extraction config loaded")
print("Style endpoint (GPT-5.2):", AZURE_OPENAI_STYLE_URL[:100] + "...")
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Found PDF: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\emirates_nbd_group_2024_ifrs_s1_s2.pdf
PDF_PATH: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\emirates_nbd_group_2024_ifrs_s1_s2.pdf
RAW_OUTPUT_PATH: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_style_reference_raw.json
CLEAN_OUTPUT_PATH: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_style_reference_clean.json
DEBUG: True
Azure REST style extraction config loaded
Style endpoint (GPT-5.2): https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.2/chat/completions...
API key loaded: True


## Azure REST configuration check


In [4]:
# Same configuration check style as the generation notebook.
# Keys are never printed.

style_config_status = {
    "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
    "style_url_loaded": bool(AZURE_OPENAI_STYLE_URL),
    "uses_judge_url_fallback": (
        bool(os.getenv("AZURE_OPENAI_JUDGE_URL"))
        and AZURE_OPENAI_STYLE_URL == _clean_url(os.getenv("AZURE_OPENAI_JUDGE_URL"))
    ),
    "style_url_preview": AZURE_OPENAI_STYLE_URL[:100] + "..." if AZURE_OPENAI_STYLE_URL else None,
}

print(json.dumps(style_config_status, indent=2))

{
  "shared_key_loaded": true,
  "style_url_loaded": true,
  "uses_judge_url_fallback": true,
  "style_url_preview": "https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.2/chat/completions..."
}


In [5]:
# ── Section detection config ────────────────────────────────
# For this specific Emirates NBD PDF, fixed page ranges are more reliable than keyword grouping.
# Page numbers are PDF-visible page numbers / one-indexed PDF pages.

USE_FIXED_EMIRATES_PAGE_RANGES = True

EMIRATES_SECTION_PAGE_RANGES = {
    # Skip page 4 because it is a section divider image page.
    "general": (5, 7),

    # Skip page 8 because it is a section divider image page.
    "governance": (9, 18),

    # Skip page 19 because it is a section divider image page.
    "strategy": (20, 46),

    # Skip page 47 because it is a section divider image page.
    "risk_management": (48, 54),

    # Skip page 55 because it is a section divider image page.
    "metrics_and_targets": (56, 63),
}

SECTION_ORDER = [
    "general",
    "governance",
    "strategy",
    "risk_management",
    "metrics_and_targets",
]

SECTION_DISPLAY_NAMES = {
    "general": "General Requirements",
    "governance": "Governance",
    "strategy": "Strategy",
    "risk_management": "Risk management",
    "metrics_and_targets": "Metrics and targets",
}

# Used only for fallback/debug.
SECTION_KEYWORDS = {
    "governance": [
        "governance", "board oversight", "board of directors",
        "management role", "committee", "oversight"
    ],
    "strategy": [
        "strategy", "strategic", "climate-related risks", "opportunities",
        "scenario analysis", "transition plan", "resilience",
        "time horizon", "physical risk", "transition risk"
    ],
    "risk_management": [
        "risk management", "risk identification", "risk assessment",
        "enterprise risk", "climate risk integration", "risk appetite",
        "risk framework"
    ],
    "metrics_and_targets": [
        "metrics", "targets", "scope 1", "scope 2", "scope 3",
        "ghg emissions", "carbon", "tco2e", "net zero",
        "financed emissions", "baseline year", "emissions intensity"
    ]
}

SKIP_PAGE_PATTERNS = [
    r"^table of contents",
    r"^contents$",
    r"^appendix",
    r"^\d+$",
]

MAX_CHUNK_WORDS = 3600

print("Fixed Emirates section ranges ready")
print(json.dumps(EMIRATES_SECTION_PAGE_RANGES, indent=2))

Fixed Emirates section ranges ready
{
  "general": [
    5,
    7
  ],
  "governance": [
    9,
    18
  ],
  "strategy": [
    20,
    46
  ],
  "risk_management": [
    48,
    54
  ],
  "metrics_and_targets": [
    56,
    63
  ]
}


In [6]:
# ── Step 1: PDF page extraction and cleaning ────────────────
def clean_pdf_text(text: str) -> str:
    """
    Light text cleaner. PyMuPDF already preserves spaces well for this PDF.
    """
    if not text:
        return ""

    text = text.replace("\x08", " ")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u2002", " ")
    text = text.replace("\u2003", " ")
    text = text.replace("￾", "-")

    # Normalise line endings and repeated whitespace, but preserve paragraph lines.
    cleaned_lines = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _extract_pages_with_pymupdf(pdf_path: Path) -> list[dict]:
    """
    Primary extractor for this Emirates NBD PDF.
    PyMuPDF's text layer is much cleaner than pdfplumber for this file.
    """
    if fitz is None:
        return []

    pages = []
    doc = fitz.open(str(pdf_path))

    for i, page in enumerate(tqdm(doc, desc="Extracting pages with PyMuPDF")):
        text = page.get_text("text", sort=True) or ""
        text = clean_pdf_text(text)

        if not text or len(text) < 50:
            continue

        pages.append({
            "page_num": i + 1,
            "text": text,
            "word_count": len(text.split()),
            "extractor": "pymupdf_text_sort_true",
        })

    doc.close()
    return pages


def _extract_pages_with_pdfplumber_words(pdf_path: Path) -> list[dict]:
    """
    Fallback extractor only. For this PDF, PyMuPDF is preferred.
    """
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(tqdm(pdf.pages, desc="Fallback extracting pages with pdfplumber words")):
            words = page.extract_words(
                x_tolerance=2,
                y_tolerance=3,
                keep_blank_chars=False,
                use_text_flow=False,
            )

            # Simple visual reconstruction.
            words = sorted(words, key=lambda w: (round(float(w.get("top", 0)), 1), float(w.get("x0", 0))))
            lines = []
            current = []
            current_top = None

            for w in words:
                top = float(w.get("top", 0))
                txt = str(w.get("text", "")).replace("\x08", " ").strip()
                if not txt:
                    continue

                if current_top is None or abs(top - current_top) <= 3:
                    current.append(txt)
                    current_top = top if current_top is None else current_top
                else:
                    lines.append(" ".join(current))
                    current = [txt]
                    current_top = top

            if current:
                lines.append(" ".join(current))

            text = clean_pdf_text("\n".join(lines))

            if not text or len(text) < 50:
                continue

            pages.append({
                "page_num": i + 1,
                "text": text,
                "word_count": len(text.split()),
                "extractor": "pdfplumber_words_fallback",
            })

    return pages


def looks_garbled_or_concatenated(text: str) -> bool:
    """
    Detect badly extracted text.
    We expect some company names/URLs to be long, so this is only a warning.
    """
    if not text:
        return True

    # Examples of bad extraction: completeparagraphwithoutspaces or GGeenneerraall.
    very_long_alpha_tokens = re.findall(r"\b[A-Za-z]{28,}\b", text)
    repeated_heading_pairs = len(re.findall(r"([A-Za-z])\1{2,}", text))

    return len(very_long_alpha_tokens) >= 3 or repeated_heading_pairs > 30


def is_table_of_contents_page(text: str) -> bool:
    """
    Detect TOC/Contents pages.
    For this PDF, page 3 is the main contents page.
    """
    t = clean_pdf_text(text).lower()
    first_2500 = t[:2500]

    toc_signals = [
        "contents",
        "general requirements",
        "governance",
        "strategy",
        "risk management",
        "metrics and targets",
        "appendix",
    ]

    signal_count = sum(s in first_2500 for s in toc_signals)
    has_many_section_numbers = len(re.findall(r"\b[1-5]\.\d+\b", first_2500)) >= 8

    return ("contents" in first_2500 and signal_count >= 4) or has_many_section_numbers


def is_section_divider_page(page_num: int, text: str) -> bool:
    """
    Divider pages are visual title pages: 4, 8, 19, 47, 55, 64, 67.
    They are useful for structure but not for style extraction.
    """
    known_dividers = {1, 2, 4, 8, 19, 47, 55, 64, 67}
    if page_num in known_dividers:
        return True

    t = clean_pdf_text(text).lower()
    if len(t.split()) <= 8 and any(label.lower() in t for label in SECTION_DISPLAY_NAMES.values()):
        return True

    return False


def should_skip_page(page: dict, debug: bool = False) -> bool:
    """
    Skip cover, TOC, section divider, appendix cover, and very low-content pages.
    """
    page_num = page["page_num"]
    text = clean_pdf_text(page["text"])

    if not text:
        return True

    if is_table_of_contents_page(text):
        return True

    if is_section_divider_page(page_num, text):
        return True

    first_line = text.split("\n")[0].lower().strip()
    if any(re.match(p, first_line) for p in SKIP_PAGE_PATTERNS):
        return True

    return False


def extract_pdf_pages(pdf_path: str | Path, debug: bool = False) -> list[dict]:
    """
    Extract pages from the uploaded Emirates NBD PDF.
    """
    pdf_path = Path(pdf_path)
    log.info(f"Opening PDF: {pdf_path}")

    pages = _extract_pages_with_pymupdf(pdf_path)

    if not pages:
        log.warning("PyMuPDF extraction unavailable or empty. Falling back to pdfplumber.")
        pages = _extract_pages_with_pdfplumber_words(pdf_path)

    before_skip = len(pages)
    pages = [p for p in pages if not should_skip_page(p, debug=debug)]

    log.info(f"Extracted {len(pages)} content pages after skipping ({before_skip - len(pages)} skipped)")
    if pages:
        log.info(f"Extractor used: {pages[0].get('extractor')}")

    bad_pages = [p["page_num"] for p in pages if looks_garbled_or_concatenated(p["text"])]
    if bad_pages:
        print("WARNING: possible concatenated/garbled text on pages:", bad_pages[:20])

    return pages


print("PyMuPDF-first PDF extraction ready")

PyMuPDF-first PDF extraction ready


In [7]:
# ── Run Step 1: Extract pages ───────────────────────────────
if PDF_PATH is None:
    raise FileNotFoundError(
        "PDF_PATH is None. Set PDF_PATH_MANUAL in the Configuration cell, "
        "or set EMIRATES_NBD_PDF_PATH in your .env file."
    )

pages = extract_pdf_pages(PDF_PATH, debug=DEBUG)

print("Pages extracted:", len(pages))
print("First 5 extracted pages:")
for page in pages[:5]:
    first = page["text"].splitlines()[0] if page["text"].splitlines() else ""
    print(f"- page {page['page_num']} | words={page['word_count']} | extractor={page.get('extractor')} | first line={first[:120]}")

print("\nSample text quality check:")
for page in pages[:3]:
    print("\n" + "="*70)
    print(f"PAGE {page['page_num']}")
    print("="*70)
    print(page["text"][:900])

2026-06-12 10:08:13,239 [INFO] Opening PDF: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\emirates_nbd_group_2024_ifrs_s1_s2.pdf
Extracting pages with PyMuPDF: 100%|██████████| 69/69 [00:02<00:00, 25.11it/s]
2026-06-12 10:08:16,199 [INFO] Extracted 54 content pages after skipping (6 skipped)
2026-06-12 10:08:16,200 [INFO] Extractor used: pymupdf_text_sort_true


Pages extracted: 54
First 5 extracted pages:
- page 5 | words=552 | extractor=pymupdf_text_sort_true | first line=General Requirements
- page 6 | words=466 | extractor=pymupdf_text_sort_true | first line=General Requirements
- page 7 | words=529 | extractor=pymupdf_text_sort_true | first line=GeneralGeneral RequirementsRequirements
- page 9 | words=475 | extractor=pymupdf_text_sort_true | first line=Governance
- page 10 | words=518 | extractor=pymupdf_text_sort_true | first line=Governance

Sample text quality check:

PAGE 5
General Requirements
General Requirements
1.1 Understanding Emirates NBD group’s approach towards implementing IFRS S1 and IFRS S2 1.2 Fair presentation
requirements
The information disclosed within this report
IFRS S1 General Requirements for Disclosure of opportunities that could reasonably impact their future, corresponds to the Financial Year 2024. This information
Sustainability-related Financial Information and including effects on cash flows, access to finan

In [8]:
# ── Step 2: Fixed page-range section grouping ───────────────
def classify_page_section(page_text: str) -> Optional[str]:
    """
    Fallback classifier based on keyword density.
    Fixed ranges are used for the actual Emirates NBD PDF.
    """
    text_lower = clean_pdf_text(page_text).lower()
    scores = {}

    for section, keywords in SECTION_KEYWORDS.items():
        score = sum(text_lower.count(kw) for kw in keywords)
        if score > 0:
            scores[section] = score

    if not scores:
        return None

    return max(scores, key=scores.get)


def classify_page_scores(page_text: str) -> dict:
    """
    Debug helper: return all keyword scores.
    """
    text_lower = clean_pdf_text(page_text).lower()
    return {
        section: sum(text_lower.count(kw) for kw in keywords)
        for section, keywords in SECTION_KEYWORDS.items()
    }


def group_pages_by_fixed_ranges(pages: list[dict]) -> dict[str, list[dict]]:
    """
    Group pages using known Emirates NBD report ranges from the Contents page.
    """
    pages_by_num = {p["page_num"]: p for p in pages}
    sections = {s: [] for s in SECTION_ORDER}
    sections["unclassified"] = []

    assigned = set()

    for section, (start, end) in EMIRATES_SECTION_PAGE_RANGES.items():
        selected = []
        for page_num in range(start, end + 1):
            page = pages_by_num.get(page_num)
            if page is not None:
                selected.append(page)
                assigned.add(page_num)
        sections[section] = selected

    sections["unclassified"] = [
        p for p in pages
        if p["page_num"] not in assigned
    ]

    return sections


def group_pages_by_keyword_fallback(pages: list[dict]) -> dict[str, list[dict]]:
    """
    Fallback only if you use another PDF without the Emirates page ranges.
    """
    sections = {s: [] for s in SECTION_ORDER}
    sections["unclassified"] = []

    for page in pages:
        section = classify_page_section(page["text"])
        if section:
            sections[section].append(page)
        else:
            sections["unclassified"].append(page)

    return sections


def group_pages_by_section(pages: list[dict]) -> dict[str, list[dict]]:
    """
    Main grouping function used by the notebook.
    """
    if USE_FIXED_EMIRATES_PAGE_RANGES:
        sections = group_pages_by_fixed_ranges(pages)
        print("Using fixed Emirates page ranges from the Contents page.")
    else:
        sections = group_pages_by_keyword_fallback(pages)
        print("Using keyword fallback grouping.")

    print("\nSection page ranges/counts:")
    for name, pages_list in sections.items():
        if pages_list:
            page_nums = [p["page_num"] for p in pages_list]
            print(f"- {name}: {len(pages_list)} pages | {min(page_nums)}-{max(page_nums)} | pages={page_nums[:12]}{'...' if len(page_nums) > 12 else ''}")
        else:
            print(f"- {name}: 0 pages")

    return sections


print("Fixed page-range section grouping ready")

Fixed page-range section grouping ready


In [9]:
# ── Run Step 2: Group pages by section ──────────────────────
section_pages = group_pages_by_section(pages)

print("\nSection page counts:")
for section, pg_list in section_pages.items():
    print(f"- {section}: {len(pg_list)} pages")

print("\nPage quality debug sample:")
for page in pages[:10]:
    first = page["text"].splitlines()[0][:90] if page["text"].splitlines() else ""
    scores = classify_page_scores(page["text"])
    print(f"page {page['page_num']:>3} | scores={scores} | first='{first}'")

Using fixed Emirates page ranges from the Contents page.

Section page ranges/counts:
- general: 3 pages | 5-7 | pages=[5, 6, 7]
- governance: 10 pages | 9-18 | pages=[9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
- strategy: 27 pages | 20-46 | pages=[20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]...
- risk_management: 7 pages | 48-54 | pages=[48, 49, 50, 51, 52, 53, 54]
- metrics_and_targets: 7 pages | 56-62 | pages=[56, 57, 58, 59, 60, 61, 62]
- unclassified: 0 pages

Section page counts:
- general: 3 pages
- governance: 10 pages
- strategy: 27 pages
- risk_management: 7 pages
- metrics_and_targets: 7 pages
- unclassified: 0 pages

Page quality debug sample:
page   5 | scores={'governance': 2, 'strategy': 9, 'risk_management': 1, 'metrics_and_targets': 2} | first='General Requirements'
page   6 | scores={'governance': 0, 'strategy': 1, 'risk_management': 0, 'metrics_and_targets': 1} | first='General Requirements'
page   7 | scores={'governance': 3, 'strategy': 4, 'risk_management': 0, 'met

In [10]:
# ── Step 3: Build section chunks with first/middle/last sampling ─────
def remove_likely_headers_footers(text: str) -> str:
    """
    Remove recurring low-value page headers/footers from chunks.
    """
    cleaned = []

    for line in text.splitlines():
        l = line.strip()
        ll = l.lower()

        if not l:
            continue
        if re.fullmatch(r"\d+", l):
            continue
        if ll.startswith("emirates nbd group 2024 ifrs s1 and s2 report"):
            continue
        if ll in {"governance", "strategy", "risk management", "metrics and targets", "general requirements"}:
            # Avoid repeated page headers. A clean section label is added by build_section_chunk().
            continue

        cleaned.append(l)

    return "\n".join(cleaned)


def _sample_words_first_middle_last(words: list[str], max_words: int) -> list[str]:
    """
    For long sections, sample first/middle/last to avoid missing later material.

    This fixes the earlier problem where Strategy and Metrics only captured opening pages,
    so scenario analysis and financed-emissions wording were missed.
    """
    n = len(words)
    if n <= max_words:
        return words

    # Divide the allowance across first, middle, and last.
    first_n = int(max_words * 0.40)
    middle_n = int(max_words * 0.30)
    last_n = max_words - first_n - middle_n

    first = words[:first_n]

    mid_center = n // 2
    mid_start = max(first_n, mid_center - middle_n // 2)
    mid_end = min(n - last_n, mid_start + middle_n)
    middle = words[mid_start:mid_end]

    last = words[-last_n:]

    return (
        first
        + ["\n\n[...middle of section sampled below...]\n\n"]
        + middle
        + ["\n\n[...end of section sampled below...]\n\n"]
        + last
    )


def build_section_chunk(
    pages: list[dict],
    section: str | None = None,
    max_words: int = MAX_CHUNK_WORDS,
) -> str:
    """
    Concatenate cleaned section pages and sample long sections across beginning, middle, and end.
    """
    if not pages:
        return ""

    combined_text = "\n\n".join(clean_pdf_text(page["text"]) for page in pages)
    combined_text = remove_likely_headers_footers(combined_text)

    words = combined_text.split()
    sampled_words = _sample_words_first_middle_last(words, max_words=max_words)
    sampled_text = " ".join(sampled_words)

    # Add a clean section label for the style model.
    if section:
        heading = SECTION_DISPLAY_NAMES.get(section, section)
        sampled_text = f"{heading}\n\n{sampled_text}"

    return sampled_text.strip()


print("build_section_chunk() ready with first/middle/last sampling")

build_section_chunk() ready with first/middle/last sampling


In [11]:
# ── Run Step 3: Build chunks ────────────────────────────────
section_chunks = {}

# Analyze only the real report sections.
sections_to_analyze = [
    "general",
    "governance",
    "strategy",
    "risk_management",
    "metrics_and_targets",
]

SECTION_WORD_BUDGETS = {
    "general": 1800,
    "governance": 3200,
    "strategy": 4200,
    "risk_management": 3600,
    "metrics_and_targets": 3600,
}

for section in sections_to_analyze:
    pg_list = section_pages.get(section, [])
    if pg_list:
        budget = SECTION_WORD_BUDGETS.get(section, MAX_CHUNK_WORDS)
        chunk = build_section_chunk(pg_list, section=section, max_words=budget)
        section_chunks[section] = chunk
        page_nums = [p["page_num"] for p in pg_list]
        print(f"{section}: {len(chunk.split())} words | budget={budget} | pages {page_nums}")
    else:
        print(f"{section}: 0 pages -> no chunk")

print("\nChunk preview:")
for section, chunk in section_chunks.items():
    print("\n" + "="*80)
    print(section.upper())
    print("="*80)
    print(chunk[:1400])

general: 1513 words | budget=1800 | pages [5, 6, 7]
governance: 3211 words | budget=3200 | pages [9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
strategy: 4211 words | budget=4200 | pages [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
risk_management: 3166 words | budget=3600 | pages [48, 49, 50, 51, 52, 53, 54]
metrics_and_targets: 3189 words | budget=3600 | pages [56, 57, 58, 59, 60, 61, 62]

Chunk preview:

GENERAL
General Requirements

1.1 Understanding Emirates NBD group’s approach towards implementing IFRS S1 and IFRS S2 1.2 Fair presentation requirements The information disclosed within this report IFRS S1 General Requirements for Disclosure of opportunities that could reasonably impact their future, corresponds to the Financial Year 2024. This information Sustainability-related Financial Information and including effects on cash flows, access to financing and is complete, neutral and accurate. While preparing this IFRS S2 Cl

## Diagnostic: chunk quality check


In [12]:
# ── Diagnostic: check chunk quality before calling Azure ─────────
def chunk_quality_report(section_chunks: dict) -> dict:
    report = {}

    for section, chunk in section_chunks.items():
        long_unspaced = re.findall(r"\b[A-Za-z]{28,}\b", chunk)
        repeated_artifacts = re.findall(r"([A-Za-z])\1{2,}", chunk)

        report[section] = {
            "word_count": len(chunk.split()),
            "starts_with": chunk[:160],
            "long_unspaced_words_count": len(long_unspaced),
            "long_unspaced_words_sample": long_unspaced[:10],
            "repeated_letter_artifact_count": len(repeated_artifacts),
            "looks_like_table_of_contents": is_table_of_contents_page(chunk),
            "looks_garbled_or_concatenated": looks_garbled_or_concatenated(chunk),
        }

    return report

quality = chunk_quality_report(section_chunks)
print(json.dumps(quality, indent=2, ensure_ascii=False))

{
  "general": {
    "word_count": 1513,
    "starts_with": "General Requirements\n\n1.1 Understanding Emirates NBD group’s approach towards implementing IFRS S1 and IFRS S2 1.2 Fair presentation requirements The informatio",
    "long_unspaced_words_count": 0,
    "long_unspaced_words_sample": [],
    "repeated_letter_artifact_count": 0,
    "looks_like_table_of_contents": false,
    "looks_garbled_or_concatenated": false
  },
  "governance": {
    "word_count": 3211,
    "starts_with": "Governance\n\nThe Board of Directors Asset Management ESG governance Oversees the Group’s ESG and climate strategies and guides the group towards its sustainabili",
    "long_unspaced_words_count": 0,
    "long_unspaced_words_sample": [],
    "repeated_letter_artifact_count": 0,
    "looks_like_table_of_contents": false,
    "looks_garbled_or_concatenated": false
  },
  "strategy": {
    "word_count": 4211,
    "starts_with": "Strategy\n\n3.1 Overview Sustainability is embedded in the Group’s culture,

In [13]:
# ── Step 4: LLM style extraction prompts ────────────────────
# These prompts extract abstract style patterns only.
# They explicitly avoid creating agent prompts or copying Emirates-specific language.

SECTION_ANALYSIS_BASE = """
You are analysing a reference IFRS S1/S2 bank report section.

Return valid JSON only.

Important rules:
- Extract writing style, organization, structure, tone, formatting, and disclosure patterns.
- Do NOT produce final generation prompts.
- Do NOT copy long phrases from the reference report.
- Do NOT write instructions that make another model speak as Emirates NBD.
- Treat Emirates NBD as reference style only, not as the future report identity.
- Convert exact wording into reusable abstract patterns where possible.
- Include short examples only when they are generic and useful.
- Keep the output concise but complete.

TEXT:
{text}
"""

SECTION_PROMPTS = {
    "general": SECTION_ANALYSIS_BASE + """
Extract the General Requirements / Reporting Basis style.

Return JSON with exactly these keys:
{
  "section": "general",
  "purpose": "...",
  "report_voice": "...",
  "organization_reference_pattern": "...",
  "typical_subsections": [],
  "opening_pattern": "...",
  "compliance_and_reporting_basis_language": [],
  "boundary_and_disclaimer_patterns": [],
  "cross_reference_patterns": [],
  "materiality_language_patterns": [],
  "entity_scope_language_patterns": [],
  "paragraph_and_list_style": "...",
  "do_not_reuse_exact_identity_terms": []
}
""",

    "governance": SECTION_ANALYSIS_BASE + """
Extract the Governance writing style.

Return JSON with exactly these keys:
{
  "section": "governance",
  "purpose": "...",
  "tone_descriptors": [],
  "typical_subsections": [],
  "opening_pattern": "...",
  "board_and_committee_language_patterns": [],
  "management_responsibility_patterns": [],
  "skills_training_remuneration_patterns": [],
  "control_and_evidence_boundary_patterns": [],
  "sentence_structure": "...",
  "tense_and_voice": "...",
  "formatting_patterns": [],
  "do_not_reuse_exact_identity_terms": []
}
""",

    "strategy": SECTION_ANALYSIS_BASE + """
Extract the Strategy writing style.

Return JSON with exactly these keys:
{
  "section": "strategy",
  "purpose": "...",
  "tone_descriptors": [],
  "typical_subsections": [],
  "opening_pattern": "...",
  "value_chain_language_patterns": [],
  "risk_opportunity_language_patterns": [],
  "time_horizon_language_patterns": [],
  "scenario_and_resilience_language_patterns": [],
  "financial_effects_language_patterns": [],
  "forward_looking_language_patterns": [],
  "sentence_structure": "...",
  "formatting_patterns": [],
  "do_not_reuse_exact_identity_terms": []
}
""",

    "risk_management": SECTION_ANALYSIS_BASE + """
Extract the Risk Management writing style.

Return JSON with exactly these keys:
{
  "section": "risk_management",
  "purpose": "...",
  "tone_descriptors": [],
  "typical_subsections": [],
  "opening_pattern": "...",
  "risk_identification_assessment_patterns": [],
  "upstream_internal_downstream_patterns": [],
  "climate_risk_management_patterns": [],
  "controls_processes_and_monitoring_patterns": [],
  "governance_link_patterns": [],
  "data_methodology_limitation_patterns": [],
  "sentence_structure": "...",
  "formatting_patterns": [],
  "do_not_reuse_exact_identity_terms": []
}
""",

    "metrics_and_targets": SECTION_ANALYSIS_BASE + """
Extract the Metrics and Targets writing style.

Return JSON with exactly these keys:
{
  "section": "metrics_and_targets",
  "purpose": "...",
  "tone_descriptors": [],
  "typical_subsections": [],
  "opening_pattern": "...",
  "metrics_table_patterns": [],
  "unit_and_number_formatting_patterns": [],
  "targets_language_patterns": [],
  "financed_emissions_language_patterns": [],
  "methodology_and_assurance_patterns": [],
  "data_limitation_patterns": [],
  "sentence_structure": "...",
  "formatting_patterns": [],
  "do_not_reuse_exact_identity_terms": []
}
"""
}

print("Abstract style extraction prompts ready")

Abstract style extraction prompts ready


In [14]:
# ── Step 5: Azure OpenAI REST API helper ────────────────────
# Same REST-call logic as the generation notebook.

def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list[dict],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: float | None = None,
    use_max_completion_tokens: bool = True,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 4,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Same behaviour as generation notebook:
    - Uses full Azure deployment URL directly.
    - Retries transient 500/502/503/504 and connection errors.
    - For GPT-5.x gateways, first tries max_completion_tokens.
    - If needed, falls back to max_tokens.
    - Does not expose API keys in errors.
    """

    preferred_field = (
        "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    )
    token_fields = [preferred_field]
    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; "
                        f"retrying attempt {attempt + 1}/{max_attempts} "
                        f"in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(
        f"{request_label} request failed for an unknown reason."
    )


def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )

    return content


def _extract_json_object(text: str) -> str:
    """
    Extract the outermost JSON object from model output.
    Handles accidental markdown fences or leading/trailing commentary.
    """
    text = str(text or "").strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")

    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def call_style_llm_json(
    system_prompt: str,
    user_prompt: str,
    *,
    max_output_tokens: int = 2800,
    request_label: str = "GPT-5.2 style extraction",
) -> dict:
    """
    GPT-5.2 style extraction with JSON-safe retry/repair.
    Uses exactly the same full deployment URL REST pattern as the generation notebook.
    """
    data = _azure_chat_completion(
        url=AZURE_OPENAI_STYLE_URL,
        api_key=AZURE_OPENAI_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=max_output_tokens,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=True,
        request_label=request_label,
    )

    content = _extract_message_content(data)
    candidate = _extract_json_object(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        print(f"{request_label} returned incomplete/invalid JSON. Attempting JSON repair...")

        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "Preserve the original meaning. "
            "Keep strings concise. Do not add markdown fences or commentary."
        )

        repair_user = f"""
Repair the following malformed or truncated output into one complete valid JSON object.

Output to repair:
{content}
"""

        repair_data = _azure_chat_completion(
            url=AZURE_OPENAI_STYLE_URL,
            api_key=AZURE_OPENAI_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=2200,
            use_max_completion_tokens=True,
            temperature=None,
            json_mode=True,
            request_label=f"{request_label} JSON repair",
        )

        repaired = _extract_json_object(_extract_message_content(repair_data))
        return json.loads(repaired)


def get_azure_openai_client() -> dict:
    """
    Kept only so existing cells can still do:
    client = get_azure_openai_client()

    It returns the already validated REST config.
    """
    validate_style_config()
    return {
        "url": AZURE_OPENAI_STYLE_URL,
        "api_key_loaded": bool(AZURE_OPENAI_API_KEY),
    }


def analyze_section_with_azure(
    client: dict,
    section_name: str,
    text_chunk: str,
    debug: bool = False
) -> Optional[dict]:
    """
    Send a section chunk to GPT-5.2 using the same REST URL logic as the generation notebook.
    """
    if not text_chunk.strip():
        log.warning(f"Empty chunk for section '{section_name}' — skipping")
        return None

    prompt_template = SECTION_PROMPTS.get(section_name, SECTION_PROMPTS["general"])
    prompt = fill_prompt_template(prompt_template, text=text_chunk)

    if debug:
        log.info(f"Sending {len(text_chunk.split())} words to GPT-5.2 style endpoint for '{section_name}'")

    try:
        parsed = call_style_llm_json(
            system_prompt=(
                "You are a document style analyst. "
                "You always respond with valid JSON only — no markdown, no preamble, no explanation. "
                "If you cannot extract a field, use null. Never invent data not present in the text."
            ),
            user_prompt=prompt,
            max_output_tokens=2800,
            request_label=f"GPT-5.2 style extraction: {section_name}",
        )

        log.info(f"  ✓ '{section_name}' analysis complete")
        return parsed

    except Exception as e:
        log.error(f"Azure GPT-5.2 style extraction error for '{section_name}': {e}")
        return None




def fill_prompt_template(template: str, **values) -> str:
    """
    Safe prompt filler.

    Do NOT use str.format() for these prompts because they contain JSON examples like:
    {
      "section": "..."
    }

    str.format() treats those JSON braces as placeholders and raises KeyError.
    This function only replaces explicit placeholders such as {text} and {section_analyses}.
    """
    out = template
    for key, value in values.items():
        out = out.replace("{" + key + "}", str(value))
    return out

print("Azure REST helper functions ready — same logic as generation notebook")

Azure REST helper functions ready — same logic as generation notebook


## Prompt formatting fix

This notebook uses `fill_prompt_template()` instead of Python `.format()` because the prompts contain JSON examples with `{}` braces. This avoids `KeyError: '\n  "section"'`.


In [15]:
# ── Run Step 5: Analyze one section first for debugging ─────
# Start with one section before running all sections.
# Change TEST_SECTION if needed.

TEST_SECTION = "governance"

client = get_azure_openai_client()

test_analysis = analyze_section_with_azure(
    client=client,
    section_name=TEST_SECTION,
    text_chunk=section_chunks.get(TEST_SECTION, ""),
    debug=DEBUG,
)

print(json.dumps(test_analysis, indent=2, ensure_ascii=False) if test_analysis else "No analysis returned")

2026-06-12 10:08:16,509 [INFO] Sending 3211 words to GPT-5.2 style endpoint for 'governance'
2026-06-12 10:08:41,843 [INFO]   ✓ 'governance' analysis complete


{
  "section": "governance",
  "purpose": "Describe the governance model for sustainability and climate-related matters by mapping oversight across board, board committees, executive management, working groups, and subsidiaries; clarify decision rights, reporting lines, meeting cadence, and how sustainability is embedded into risk, credit, product governance, and disclosures.",
  "tone_descriptors": [
    "formal",
    "institutional",
    "assurance-oriented",
    "risk-governance focused",
    "process-driven",
    "regulatory-aligned",
    "accountability-centric"
  ],
  "typical_subsections": [
    "overview of governance structure and decision-making flow",
    "role of the board of directors",
    "role of nominated board committees (e.g., ESG/nomination/remuneration; risk committee)",
    "executive management committees and approvals (e.g., executive committee, risk committee)",
    "roles of key executives (e.g., sustainability lead; risk lead) and committee chairing",
    "wo

In [16]:
# ── Run Step 5b: Analyze all sections ───────────────────────
# Run after the test section works.

section_analyses = {}

for section, chunk in section_chunks.items():
    print("\n" + "="*80)
    print("Analyzing:", section)
    print("="*80)

    analysis = analyze_section_with_azure(
        client=client,
        section_name=section,
        text_chunk=chunk,
        debug=DEBUG,
    )
    section_analyses[section] = analysis

print("\nCompleted analyses:")
for section, analysis in section_analyses.items():
    print(f"- {section}: {'OK' if analysis else 'FAILED'}")

2026-06-12 10:08:41,863 [INFO] Sending 1513 words to GPT-5.2 style endpoint for 'general'



Analyzing: general


2026-06-12 10:09:03,136 [INFO]   ✓ 'general' analysis complete
2026-06-12 10:09:03,142 [INFO] Sending 3211 words to GPT-5.2 style endpoint for 'governance'



Analyzing: governance


2026-06-12 10:09:21,508 [INFO]   ✓ 'governance' analysis complete
2026-06-12 10:09:21,510 [INFO] Sending 4211 words to GPT-5.2 style endpoint for 'strategy'


...


Analyzing: risk_management


2026-06-12 10:10:11,671 [INFO]   ✓ 'risk_management' analysis complete
2026-06-12 10:10:11,677 [INFO] Sending 3189 words to GPT-5.2 style endpoint for 'metrics_and_targets'



Analyzing: metrics_and_targets


2026-06-12 10:10:37,476 [INFO]   ✓ 'metrics_and_targets' analysis complete



Completed analyses:
- general: OK
- governance: OK
- strategy: OK
- risk_management: OK
- metrics_and_targets: OK


## Unified clean style guide prompt


In [17]:
# ── Step 6: Deterministic clean style-guide synthesis ───────
def _as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [v for v in value if v not in [None, ""]]
    if isinstance(value, str) and value.strip():
        return [value.strip()]
    return []


def _dedupe_keep_order(items):
    seen = set()
    out = []
    for item in items:
        if item is None:
            continue
        item_str = str(item).strip()
        if not item_str:
            continue
        key = item_str.lower()
        if key not in seen:
            out.append(item_str)
            seen.add(key)
    return out


def _get_nested(dct, *keys, default=None):
    cur = dct
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def _collect(section_analyses: dict, section: str, *keys) -> list:
    analysis = section_analyses.get(section, {})
    collected = []
    for key in keys:
        collected.extend(_as_list(analysis.get(key)))
    return _dedupe_keep_order(collected)


def _shorten(items, max_items=8, max_chars=260):
    out = []
    for item in items:
        item = str(item).strip()
        if len(item) > max_chars:
            item = item[:max_chars].rstrip() + "..."
        out.append(item)
        if len(out) >= max_items:
            break
    return out


def synthesize_style_guide(client, section_analyses: dict) -> dict:
    """
    Deterministic synthesis.

    This intentionally avoids a second GPT call because the previous LLM synthesis
    returned incomplete/invalid JSON. It builds a clean, reusable style guide from
    already-valid per-section JSON analyses.
    """
    log.info("Synthesizing unified style guide deterministically...")

    valid = {
        k: v for k, v in section_analyses.items()
        if isinstance(v, dict) and v
    }

    if not valid:
        log.error("No valid section analyses to synthesize.")
        return {}

    all_tones = []
    for analysis in valid.values():
        all_tones.extend(_as_list(analysis.get("tone_descriptors")))
    all_tones = _dedupe_keep_order(all_tones)

    structure_patterns = {}
    for section in ["general", "governance", "strategy", "risk_management", "metrics_and_targets"]:
        analysis = valid.get(section, {})
        structure_patterns[section] = _shorten(
            _as_list(analysis.get("typical_subsections"))
            + _as_list(analysis.get("opening_pattern"))
            + _as_list(analysis.get("paragraph_and_list_style"))
            + _as_list(analysis.get("formatting_patterns")),
            max_items=10,
        )

    language_patterns = {
        "compliance_and_reporting_basis": _shorten(
            _collect(
                valid, "general",
                "compliance_and_reporting_basis_language",
                "boundary_and_disclaimer_patterns",
                "cross_reference_patterns",
                "entity_scope_language_patterns",
            ),
            max_items=12,
        ),
        "governance_oversight": _shorten(
            _collect(
                valid, "governance",
                "board_and_committee_language_patterns",
                "management_responsibility_patterns",
                "skills_training_remuneration_patterns",
                "control_and_evidence_boundary_patterns",
            ),
            max_items=14,
        ),
        "strategy_value_chain": _shorten(
            _collect(
                valid, "strategy",
                "value_chain_language_patterns",
                "risk_opportunity_language_patterns",
                "time_horizon_language_patterns",
                "scenario_and_resilience_language_patterns",
                "financial_effects_language_patterns",
                "forward_looking_language_patterns",
            ),
            max_items=16,
        ),
        "risk_management_process": _shorten(
            _collect(
                valid, "risk_management",
                "risk_identification_assessment_patterns",
                "upstream_internal_downstream_patterns",
                "climate_risk_management_patterns",
                "controls_processes_and_monitoring_patterns",
                "governance_link_patterns",
                "data_methodology_limitation_patterns",
            ),
            max_items=16,
        ),
        "metrics_targets": _shorten(
            _collect(
                valid, "metrics_and_targets",
                "metrics_table_patterns",
                "unit_and_number_formatting_patterns",
                "targets_language_patterns",
            ),
            max_items=12,
        ),
        "financed_emissions": _shorten(
            _collect(
                valid, "metrics_and_targets",
                "financed_emissions_language_patterns",
                "methodology_and_assurance_patterns",
                "data_limitation_patterns",
            ),
            max_items=12,
        ),
        "methodology_assurance_limitations": _shorten(
            _collect(valid, "metrics_and_targets", "methodology_and_assurance_patterns", "data_limitation_patterns")
            + _collect(valid, "risk_management", "data_methodology_limitation_patterns")
            + _collect(valid, "general", "boundary_and_disclaimer_patterns"),
            max_items=14,
        ),
        "evidence_boundary": [
            "When evidence is unavailable, state the limitation using report-style wording rather than inventing details.",
            "Use boundary statements for missing governance charters, controls, methodology detail, assurance coverage, comparative information, and forward-looking estimates.",
            "Distinguish source-data indicators from independently assured or fully implemented processes.",
        ],
    }

    def section_guidance(section, default_purpose, flow):
        analysis = valid.get(section, {})
        return {
            "purpose": analysis.get("purpose") or default_purpose,
            "recommended_flow": flow,
            "style_notes": _shorten(
                _as_list(analysis.get("sentence_structure"))
                + _as_list(analysis.get("tense_and_voice"))
                + _as_list(analysis.get("formatting_patterns"))
                + _as_list(analysis.get("paragraph_and_list_style")),
                max_items=8,
            ),
        }

    style_guide = {
        "style_reference_version": "clean_v1_deterministic",
        "reference_report_role": "style_reference_only",
        "identity_policy": {
            "rule": "Use the target bank identity from the payload. Never write as Emirates NBD or imply the target bank is Emirates NBD.",
            "recommended_target_terms": [
                "the Bank",
                "the Group",
                "the institution",
                "the reporting entity",
                "the target bank name from payload",
            ],
            "forbidden_reference_identity_terms": [
                "Emirates NBD",
                "Emirates NBD Group",
                "DenizBank",
                "Emirates Islamic",
                "BNRESGC",
                "BRC",
                "KPMG Lower Gulf",
                "named Emirates executives from the reference report",
            ],
        },
        "global_style_rules": [
            "Write in a formal, technical and structured IFRS S1/S2 bank-reporting voice.",
            "Use mostly third-person institutional language, with first-person plural only for commitments or process descriptions when appropriate.",
            "Prefer clear institutional actors and accountability verbs such as oversees, approves, reviews, monitors, is responsible for, and is governed by.",
            "Use medium-length paragraphs and introduce lists with clear colon-led lead-ins.",
            "Anchor claims to evidence, policies, committees, frameworks, metrics, reporting boundaries and time horizons.",
            "Use cross-references only where they exist in the target report structure; do not invent appendix references.",
            "Avoid promotional language unless it is tied to measurable actions, targets, frameworks or evidence.",
            "Do not copy long wording from the reference report; reuse structure and tone, not exact sentences.",
        ],
        "tone_profile": {
            "primary_tone": all_tones[0] if all_tones else "formal",
            "secondary_tones": _shorten(all_tones[1:] or ["structured", "technical", "compliance-oriented"], max_items=8),
            "avoid": [
                "overly conversational wording",
                "marketing slogans",
                "unsupported assurance claims",
                "reference-bank identity leakage",
                "unqualified statements of full compliance where evidence is incomplete",
                "invented committees, charters, targets, methodologies or assurance coverage",
            ],
        },
        "structure_patterns": structure_patterns,
        "language_patterns": language_patterns,
        "formatting_conventions": {
            "headings": [
                "Use clear section and subsection headings aligned to IFRS S1/S2 pillars.",
                "Use numbered headings only if the final report structure already uses numbering.",
                "Avoid decorative or slide-like headings in narrative report sections.",
            ],
            "paragraphs": [
                "Use compact narrative paragraphs, usually one topic per paragraph.",
                "Allow longer paragraphs for methodology, reporting basis, and entity background.",
                "Keep evidence boundary statements concise and placed near the relevant disclosure.",
            ],
            "lists": [
                "Use bullets to enumerate aims, responsibilities, process steps, stakeholder groups, risk categories, and metrics.",
                "Introduce lists with colon-led lead-in sentences.",
            ],
            "tables": [
                "Use table-like structures for metrics, targets, units, sources, and current values where the generation format allows tables.",
                "For narrative-only sections, convert tables into concise grouped paragraphs.",
            ],
            "numbers_units_dates": [
                "Use explicit reporting-period language such as Financial Year 2024 or year ended 31 December 2024.",
                "Use consistent units and percentages; preserve units exactly from payload evidence.",
                "State data currency or as-of dates when provided.",
                "Separate current performance, target value, baseline, and methodology where available.",
            ],
        },
        "section_guidance": {
            "general": section_guidance(
                "general",
                "Explain reporting basis, scope, compliance approach, connected information, comparatives, entity boundary and materiality.",
                [
                    "Start with the reporting period and basis of preparation.",
                    "Explain how IFRS S1/S2 are approached and whether the disclosure is voluntary, transitional, partial or complete.",
                    "Describe reporting entity, business model, value chain and sources of guidance.",
                    "Discuss materiality process and evidence boundaries.",
                    "Close with improvement or future-reporting commitment where evidence supports it.",
                ],
            ),
            "governance": section_guidance(
                "governance",
                "Describe oversight, committee responsibilities, management roles, skills, remuneration links and governance boundaries.",
                [
                    "Open with the governance model and Board-level oversight.",
                    "Describe Board and committee responsibilities separately.",
                    "Describe management responsibility and operational accountability.",
                    "Discuss skills, training, remuneration or incentives where evidenced.",
                    "Close with controls, decisions, assurance scope and evidence boundaries.",
                ],
            ),
            "strategy": section_guidance(
                "strategy",
                "Explain sustainability and climate-related risks and opportunities across the business model, value chain, strategy, financial effects and resilience.",
                [
                    "Open with the sustainability/climate strategy and strategic priorities.",
                    "Map risks and opportunities across upstream, internal operations and downstream value chain where evidenced.",
                    "Discuss strategic decision-making and financial effects.",
                    "Discuss climate scenarios, resilience and time horizons where evidenced.",
                    "Close with limitations, assumptions and future enhancements.",
                ],
            ),
            "risk_management": section_guidance(
                "risk_management",
                "Describe how sustainability and climate risks are identified, assessed, managed, monitored and integrated into risk processes.",
                [
                    "Open with integration into enterprise risk or existing risk processes.",
                    "Discuss upstream, internal operations and downstream risk management.",
                    "Describe climate risk identification, scenario/stress testing, risk appetite, monitoring and reporting.",
                    "Link processes to governance bodies, policies, controls and lines of defence where evidenced.",
                    "Close with data, methodology and implementation boundaries.",
                ],
            ),
            "metrics_and_targets": section_guidance(
                "metrics_and_targets",
                "Present sustainability/climate metrics, targets, units, methodology, financed emissions and assurance/data limitations.",
                [
                    "Open with the role of metrics and targets under IFRS S1/S2.",
                    "Present core sustainability and climate-related metrics with units and sources.",
                    "Discuss targets, baseline, progress and time horizons where evidenced.",
                    "Explain financed emissions methodology and portfolio coverage where evidenced.",
                    "Close with data quality, methodology, assurance and boundary limitations.",
                ],
            ),
        },
        "copying_risk_controls": [
            "Do not reuse the reference bank name or named reference-bank committees.",
            "Do not reuse long sentences from the reference report.",
            "Convert reference phrases into generic patterns before generation.",
            "Use only the target bank facts, values, dates, committees and assurance scope from the payload.",
            "If the style guide conflicts with payload evidence, payload evidence wins.",
        ],
        "quality_checks_for_generated_reports": [
            "No reference-bank identity terms appear in generated output.",
            "All numerical claims are supported by payload evidence.",
            "Missing evidence is disclosed as a boundary rather than filled by assumption.",
            "Governance, strategy, risk management and metrics sections have distinct purposes and do not repeat the same paragraph.",
            "Assurance language is limited to the exact assurance scope in the payload.",
            "The output sounds like a bank IFRS S1/S2 report, not a checklist or internal prompt response.",
        ],
    }

    log.info("  ✓ Deterministic synthesis complete")
    return style_guide


style_guide = synthesize_style_guide(client, section_analyses)

print(json.dumps(style_guide, indent=2, ensure_ascii=False) if style_guide else "No style guide generated")

2026-06-12 10:10:37,533 [INFO] Synthesizing unified style guide deterministically...
2026-06-12 10:10:37,536 [INFO]   ✓ Deterministic synthesis complete


{
  "style_reference_version": "clean_v1_deterministic",
  "reference_report_role": "style_reference_only",
  "identity_policy": {
    "rule": "Use the target bank identity from the payload. Never write as Emirates NBD or imply the target bank is Emirates NBD.",
    "recommended_target_terms": [
      "the Bank",
      "the Group",
      "the institution",
      "the reporting entity",
      "the target bank name from payload"
    ],
    "forbidden_reference_identity_terms": [
      "Emirates NBD",
      "Emirates NBD Group",
      "DenizBank",
      "Emirates Islamic",
      "BNRESGC",
      "BRC",
      "KPMG Lower Gulf",
      "named Emirates executives from the reference report"
    ]
  },
  "global_style_rules": [
    "Write in a formal, technical and structured IFRS S1/S2 bank-reporting voice.",
    "Use mostly third-person institutional language, with first-person plural only for commitments or process descriptions when appropriate.",
    "Prefer clear institutional actors and a

## Validate and clean style reference


In [18]:
# ── Step 7: Validate and clean the style reference ──────────
REFERENCE_IDENTITY_TERMS = [
    "Emirates NBD",
    "DenizBank",
    "Emirates Islamic",
    "BNRESGC",
    "BRC",
    "Vijay Bains",
    "Manoj Chawla",
    "Patrick Sullivan",
    "KPMG Lower Gulf",
]

def find_reference_identity_leaks(obj, path="root") -> list[dict]:
    """
    Find reference-bank identity terms in the clean style guide.
    Some terms may be allowed only in metadata/raw analysis, not in the clean guide.
    """
    leaks = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            leaks.extend(find_reference_identity_leaks(value, f"{path}.{key}"))

    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            leaks.extend(find_reference_identity_leaks(value, f"{path}[{idx}]"))

    elif isinstance(obj, str):
        for term in REFERENCE_IDENTITY_TERMS:
            if term.lower() in obj.lower():
                leaks.append({"path": path, "term": term, "text_preview": obj[:220]})

    return leaks


def remove_agent_prompts_if_present(style_obj: dict) -> dict:
    """
    Defensive cleanup in case the LLM still returns agent_prompts.
    """
    if isinstance(style_obj, dict):
        style_obj.pop("agent_prompts", None)
        style_obj.pop("prompts", None)
        style_obj.pop("agent_prompt_templates", None)
    return style_obj


def build_final_style_reference(
    *,
    metadata: dict,
    per_section_analysis: dict,
    unified_style_guide: dict,
) -> tuple[dict, dict]:
    """
    Build raw and clean output files.
    """
    raw = {
        "metadata": metadata,
        "per_section_analysis": per_section_analysis,
        "unified_style_guide": unified_style_guide,
    }

    clean = {
        "metadata": {
            **metadata,
            "cleaned_for_generation": True,
            "agent_prompts_removed": True,
            "reference_identity_policy": "Reference report is used only for style patterns; target bank identity must come from payload.",
        },
        "style_guide": remove_agent_prompts_if_present(unified_style_guide),
        "per_section_analysis_summary": {
            section: {
                "section": analysis.get("section"),
                "purpose": analysis.get("purpose"),
                "tone_descriptors": analysis.get("tone_descriptors", []),
                "typical_subsections": analysis.get("typical_subsections", []),
                "sentence_structure": analysis.get("sentence_structure"),
            }
            for section, analysis in per_section_analysis.items()
            if isinstance(analysis, dict)
        },
    }

    return raw, clean


print("Style reference validation helpers ready")

Style reference validation helpers ready


In [19]:
# ── Step 8: Save and inspect raw + clean style reference JSON ─────────
# Run this cell after:
# 1) section_analyses is created
# 2) style_guide is created by deterministic synthesis

if "RAW_OUTPUT_PATH" not in globals():
    RAW_OUTPUT_PATH = Path("emirates_nbd_style_reference_raw.json")

if "CLEAN_OUTPUT_PATH" not in globals():
    CLEAN_OUTPUT_PATH = Path("emirates_nbd_style_reference_clean.json")

OUTPUT_PATH = CLEAN_OUTPUT_PATH

required_vars = ["section_analyses", "style_guide"]
missing_vars = [name for name in required_vars if name not in globals()]

if missing_vars:
    raise RuntimeError(
        "Cannot save yet. Missing variables: "
        + ", ".join(missing_vars)
        + ". Run the per-section analysis and deterministic synthesis cells first."
    )

if "build_final_style_reference" not in globals():
    def build_final_style_reference(*, metadata: dict, per_section_analysis: dict, unified_style_guide: dict):
        raw = {
            "metadata": metadata,
            "per_section_analysis": per_section_analysis,
            "unified_style_guide": unified_style_guide,
        }
        clean = {
            "metadata": {
                **metadata,
                "cleaned_for_generation": True,
                "agent_prompts_removed": True,
                "reference_identity_policy": (
                    "Reference report is used only for style patterns; "
                    "target bank identity must come from payload."
                ),
            },
            "style_guide": unified_style_guide,
            "per_section_analysis_summary": {
                section: {
                    "section": analysis.get("section"),
                    "purpose": analysis.get("purpose"),
                    "tone_descriptors": analysis.get("tone_descriptors", []),
                    "typical_subsections": analysis.get("typical_subsections", []),
                    "sentence_structure": analysis.get("sentence_structure"),
                }
                for section, analysis in per_section_analysis.items()
                if isinstance(analysis, dict)
            },
        }
        return raw, clean

metadata = {
    "source_pdf": str(PDF_PATH.name if "PDF_PATH" in globals() and PDF_PATH else "unknown"),
    "pages_extracted": len(pages) if "pages" in globals() else None,
    "sections_found": (
        {section: len(pg_list) for section, pg_list in section_pages.items()}
        if "section_pages" in globals()
        else {}
    ),
    "extraction_model": "GPT-5.2 via AZURE_OPENAI_STYLE_URL/AZURE_OPENAI_JUDGE_URL",
    "chunking_strategy": "fixed Emirates page ranges + first/middle/last sampling for long sections",
    "synthesis_strategy": "deterministic synthesis from per-section JSON analyses",
    "agent_prompts_removed": True,
}

raw_style_reference, clean_style_reference = build_final_style_reference(
    metadata=metadata,
    per_section_analysis=section_analyses,
    unified_style_guide=style_guide or {},
)

RAW_OUTPUT_PATH.write_text(
    json.dumps(raw_style_reference, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

CLEAN_OUTPUT_PATH.write_text(
    json.dumps(clean_style_reference, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Raw style reference saved to: {RAW_OUTPUT_PATH.resolve()}")
print(f"Clean style reference saved to: {CLEAN_OUTPUT_PATH.resolve()}")

with open(CLEAN_OUTPUT_PATH, "r", encoding="utf-8") as f:
    ref = json.load(f)

print("\nTop-level keys:", list(ref.keys()))
print("Metadata keys:", list(ref.get("metadata", {}).keys()))
print("Style guide keys:", list(ref.get("style_guide", {}).keys()))

if "find_reference_identity_leaks" in globals():
    leaks = find_reference_identity_leaks(ref.get("style_guide", {}))
    if leaks:
        print("\nWARNING: reference identity terms still appear in the clean style guide.")
        for leak in leaks[:20]:
            print(f"- {leak['term']} at {leak['path']}: {leak['text_preview']}")
    else:
        print("\nNo obvious reference-bank identity leaks detected in clean style guide.")

print("\nReady to use:", CLEAN_OUTPUT_PATH.name)

Raw style reference saved to: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_style_reference_raw.json
Clean style reference saved to: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_style_reference_clean.json

Top-level keys: ['metadata', 'style_guide', 'per_section_analysis_summary']
Metadata keys: ['source_pdf', 'pages_extracted', 'sections_found', 'extraction_model', 'chunking_strategy', 'synthesis_strategy', 'agent_prompts_removed', 'cleaned_for_generation', 'reference_identity_policy']
Style guide keys: ['style_reference_version', 'reference_report_role', 'identity_policy', 'global_style_rules', 'tone_profile', 'structure_patterns', 'language_patterns', 'formatting_conventions', 'section_guidance', 'copying_risk_controls', 'quality_checks_for_generated_reports']

- Emirates NBD at root.identity_policy.rule: Use the target bank identity from the payload. Never write as Emirates NBD or imply the target bank is Emirates NBD.
- 

## Save raw and clean style reference JSON


In [20]:
# ── Step 8: Save raw and clean style reference JSON ─────────
metadata = {
    "source_pdf": str(PDF_PATH.name if PDF_PATH else "unknown"),
    "pages_extracted": len(pages),
    "sections_found": {section: len(pg_list) for section, pg_list in section_pages.items()},
    "extraction_model": "GPT-5.2 via AZURE_OPENAI_STYLE_URL/AZURE_OPENAI_JUDGE_URL",
    "chunking_strategy": "fixed Emirates page ranges + first/middle/last sampling for long sections",
    "agent_prompts_removed": True,
}

raw_style_reference, clean_style_reference = build_final_style_reference(
    metadata=metadata,
    per_section_analysis=section_analyses,
    unified_style_guide=style_guide or {},
)

RAW_OUTPUT_PATH.write_text(
    json.dumps(raw_style_reference, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

CLEAN_OUTPUT_PATH.write_text(
    json.dumps(clean_style_reference, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Raw style reference saved to: {RAW_OUTPUT_PATH.resolve()}")
print(f"Clean style reference saved to: {CLEAN_OUTPUT_PATH.resolve()}")

# Check for leaks in the clean guide only.
leaks = find_reference_identity_leaks(clean_style_reference.get("style_guide", {}))
if leaks:
    print("\nWARNING: reference identity terms still appear in the clean style guide.")
    print("Review these before using in generation:")
    for leak in leaks[:20]:
        print(f"- {leak['term']} at {leak['path']}: {leak['text_preview']}")
else:
    print("\nNo obvious reference-bank identity leaks detected in clean style guide.")

print("\nClean style reference top-level keys:")
print(list(clean_style_reference.keys()))

Raw style reference saved to: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_style_reference_raw.json
Clean style reference saved to: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_style_reference_clean.json

Review these before using in generation:
- Emirates NBD at root.identity_policy.rule: Use the target bank identity from the payload. Never write as Emirates NBD or imply the target bank is Emirates NBD.
- Emirates NBD at root.identity_policy.forbidden_reference_identity_terms[0]: Emirates NBD
- Emirates NBD at root.identity_policy.forbidden_reference_identity_terms[1]: Emirates NBD Group
- DenizBank at root.identity_policy.forbidden_reference_identity_terms[2]: DenizBank
- Emirates Islamic at root.identity_policy.forbidden_reference_identity_terms[3]: Emirates Islamic
- BNRESGC at root.identity_policy.forbidden_reference_identity_terms[4]: BNRESGC
- BRC at root.identity_policy.forbidden_reference_identity_terms[5]: BRC
- KP